# Inspect results and failures

<a id="understand-the-result"></a>

The last proposed revision may be worse than an earlier one, or may fail to
execute. The **selected artifact** answers “what should I keep?” The run's
**history** answers “what happened along the way?” This guide shows how to read
both, using a run with an improvement, a regression, and a failed evaluation.
An unsuccessful measurement has no score; a measured wrong answer can have a
low score. [Failures](https://sentient-xyz.github.io/meta-evolve-docs/concepts/failures/) explains that distinction.

| You want to… | Use |
|---|---|
| [Read a compact outcome](#read-a-snapshot) | `summary()` |
| [Read the selected source and score](#read-the-selected-source-and-score) | `best()` and `best_trial()` |
| [See every attempt or the winning path](#see-a-regression-stay-in-history) | `trials()` and `lineage()` |
| [Understand a failed measurement](#distinguish-a-failed-measurement) | `failure` and `inspect()` |
| [Count work](#count-the-work) | `usage()` |

The setup creates `inspection_result`: a parser run with scores of 33%, 67%,
33%, then a failed evaluation. The revisions are handwritten; Python executes
and grades them. No model, SDK, or credentials are needed.



<a id="set-up-this-page"></a>
<a id="complete-source"></a>
<a id="1-install"></a>
<a id="2-define-what-is-being-improved"></a>
<a id="3-run-a-mix-of-outcomes"></a>

## Required setup for a fresh notebook

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

**Starting source and instructions.** `SEED` reads a duration's number but
ignores its unit. `CONTRACT` tells the proposer what the parser should do.

In [ ]:
import meta_evolve as meta

CONTRACT = """Implement parse_seconds(text) for whole-number durations.
Inputs contain a number followed by s, m, or h, such as '30s' or '2h'.
Return the duration in seconds as an integer. Return only Python source.
"""

SEED = '''def parse_seconds(text):
    return int(text[:-1])
'''

**Evaluation.** `evaluate` returns the fraction of six checks answered
correctly. `load_parser` executes the source locally; use this evaluator
with the reviewed, handwritten code shown here.

In [ ]:
CASES = (
    ("30s", 30),
    ("90s", 90),
    ("2m", 120),
    ("3m", 180),
    ("1h", 3600),
    ("2h", 7200),
)


def load_parser(source):
    namespace = {}
    exec(source, namespace)
    return namespace["parse_seconds"]


def evaluate(source):
    parse = load_parser(source)
    passed = 0
    for text, expected in CASES:
        actual = parse(text)
        if type(actual) is int and actual == expected:
            passed += 1
    return passed / len(CASES)


print(f"Starting score: {evaluate(SEED):.0%}")
# Output:
# Starting score: 33%

**Revisions.** `MINUTES` adds minutes support. `COMPLETE` also handles hours;
keep it for the exercise at the end.

In [ ]:
MINUTES = '''def parse_seconds(text):
    quantity = int(text[:-1])
    return quantity * 60 if text[-1] == "m" else quantity
'''

COMPLETE = '''def parse_seconds(text):
    quantity = int(text[:-1])
    seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
    return quantity * seconds_per_unit[text[-1]]
'''

**Create a history to inspect.**

A **proposer** receives the current source and returns a proposed replacement.
Its optional second argument, `context`, includes `step`: the proposal number,
starting at `1` in this greedy run. Use it to return three fixed revisions in
order: the minutes fix, the original parser, then syntactically broken source.
This deliberately scripted proposer ignores its parent so the history is easy
to reproduce and inspect.

In [ ]:
BROKEN = "def broken("
INSPECTION_REVISIONS = (MINUTES, SEED, BROKEN)


def propose_mixed_history(source, *, context):
    return INSPECTION_REVISIONS[context.step - 1]

[`meta.improve()`][meta_evolve.improve] evaluates the seed, proposes from the
best version so far, and evaluates each valid revision. Higher scalar scores
win by default. `trials=3` allows three proposal attempts after the seed.
It returns a run containing the outcomes and selected version.

In [ ]:
inspection_result = meta.improve(
    seed=SEED,
    proposer=propose_mixed_history,
    evaluator=evaluate,
    trials=3,
)

Rerunning this setup produces the same history; `context.step` starts again
for each run. The [Start here walkthrough](https://sentient-xyz.github.io/meta-evolve-docs/start-here/) explains the
shared parser.


<a id="read-a-snapshot"></a>

After running the setup, capture a compact report:

In [ ]:
inspection_snapshot = inspection_result.summary()
inspection_snapshot

The notebook shows a completed run, a baseline score of 33% and selected score
of 67% (as numeric fractions), three proposals, four evaluations including the
baseline, and one failed attempt. History is not saved durably in this example.
Completion, measurement, and improvement are separate facts. A selected seed
or a tie is not an improvement; a failed evaluation has no score.

Use `print(inspection_snapshot)` for plain text. Capture while storage is open;
the report remains displayable after storage closes. Detailed source, evidence,
and failure information remain in the explicit queries below.

<a id="inspect-a-complete-seed-run"></a>
<a id="read-the-selected-source-and-score"></a>

<a id="4-read-the-selected-source-and-score"></a>

## 1. Read the selected source and score

`best()` returns the selected artifact; its `.value` is the source string.
`best_trial()` returns the selected attempt with its measurement and evidence.
Printing the source does not execute it.

In [ ]:
selected = inspection_result.best()
print(f"Selected score: {inspection_result.best_trial().metrics['score']:.0%}")
print(selected.value, end="")
# Output:
# Selected score: 67%
# def parse_seconds(text):
#     quantity = int(text[:-1])
#     return quantity * 60 if text[-1] == "m" else quantity

The minutes revision stays selected, even though it was followed by two more
attempts. Selection is based on measured quality, not recency.

<a id="see-a-regression-stay-in-history"></a>

<a id="5-read-every-attempt-and-the-winning-lineage"></a>

## 2. Read every attempt and the winning lineage

`trials()` includes the seed and every attempted revision, including failures.
Check `failure` before reading a score: a failed evaluation has no measurement.

In [ ]:
for number, attempt in enumerate(inspection_result.trials()):
    label = "Seed" if number == 0 else f"Revision {number}"
    if attempt.failure is not None:
        print(f"{label}: {attempt.failure.kind} (no score)")
    else:
        print(f"{label}: {attempt.metrics['score']:.0%}")
# Output:
# Seed: 33%
# Revision 1: 67%
# Revision 2: 33%
# Revision 3: evaluator_failure (no score)

Revision 2 executes successfully but gives wrong answers. Its 33% score is a
valid measurement, so it stays in history as a regression from the 67% parent.
Revision 3 cannot execute, so there is no score to compare.

`lineage()` returns the attempts along the selected artifact's parent chain.
Here it contains just the seed and the minutes revision:

In [ ]:
print("Attempts in history:", len(inspection_result.trials()))
print("Versions in winning lineage:", len(inspection_result.lineage()))
# Output:
# Attempts in history: 4
# Versions in winning lineage: 2

The later revisions were proposed from the minutes revision. Neither belongs
to the winning lineage, but both remain available through `trials()`.
An attempt's `evidence` holds any observations attached by the evaluator or
proposer; this scalar evaluator attaches none. The
[feedback guide](https://sentient-xyz.github.io/meta-evolve-docs/learn/05-use-experience/) shows how to record failed checks there.

<a id="distinguish-a-mistake-from-a-failed-measurement"></a>
<a id="distinguish-a-failed-measurement"></a>

<a id="6-understand-a-failed-evaluation"></a>

## 3. Understand a failed evaluation

The malformed source reached the evaluator, where loading it raised a
`SyntaxError`. The callable boundary recorded an `evaluator_failure` with
exception details and empty metrics:

In [ ]:
failed_attempt = inspection_result.trials()[-1]
print("Failure:", failed_attempt.failure.kind)
print("Exception:", failed_attempt.failure.details["exception_type"])
print("Metrics:", dict(failed_attempt.metrics))
# Output:
# Failure: evaluator_failure
# Exception: SyntaxError
# Metrics: {}

A failed measurement is not a zero score. A source version that runs and fails
every check would score zero under our evaluator. More detailed evaluators can
report typed failures for invalid output, timeouts, or infrastructure problems.
A proposal failure can occur before any artifact is produced.

If **every** evaluation fails, there is no winner. `inspect()` gives a summary
without requiring one. Try a separate seed-only run:

In [ ]:
failed_run = meta.improve(
    seed=BROKEN,
    proposer=propose_mixed_history,
    evaluator=evaluate,
    trials=0,
)
overview = failed_run.inspect().overview
print("Rankability:", overview.rankability)
print("Selected attempt:", overview.best_trial)
print("Evaluations:", failed_run.usage().evaluations)
# Output:
# Rankability: unrankable
# Selected attempt: None
# Evaluations: 1

`trials()`, `inspect()`, and `usage()` still work. Calling `best()` or
`best_trial()` on that run raises `meta.NoSuccessfulEvaluation`.

<a id="count-the-work"></a>

<a id="7-count-the-work"></a>

## 4. Count the work

`usage()` totals the work charged to a run. The main run made three proposals
and evaluated four source versions, counting the seed and the failed evaluation.

In [ ]:
usage = inspection_result.usage()
print("Proposal attempts:", usage.trials)
print("Evaluations:", usage.evaluations)
# Output:
# Proposal attempts: 3
# Evaluations: 4

The manual `evaluate(SEED)` preview and the separate seed-only run are outside
this run's accounting.

## Change and predict

In the required setup, change the `meta.improve()` call from
`seed=SEED` to `seed=COMPLETE`, leaving the proposer and three-attempt limit
unchanged. Predict the winner, then rerun that call and the inspection cells.

The seed now scores **100%** and stays selected. The three proposals still
produce 67%, 33%, and a failed evaluation. History still contains four attempts,
but the winning lineage contains only the seed. There are still four evaluations.

## Why isn't the last attempt the selected result?

Selection follows the objective, so a later regression or failed measurement
does not replace a better measured version. Use `trials()` for all attempts
and `lineage()` for the selected version's parent chain.

For your own run, use these same queries on the result of `improve()` or `run()`.
Check for failures before indexing metrics; if nothing was measured successfully,
start with `inspect()` instead of asking for a winner.

<a id="inspect-a-model-produced-history"></a>

Next, [set limits and stopping conditions](https://sentient-xyz.github.io/meta-evolve-docs/guides/limits/) to control how
much work the experiment can do. The
[result reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/runs-and-results/) covers aggregate inspection,
comparisons, and additional queries.